<a href="https://colab.research.google.com/github/Matheus-200/python_sis_26_2/blob/Matheus_Gabriel/aula_14/matheus_silva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
## Atividade Prática 01: Coleta de Dados via API
### Documentação: [WeatherAPI](https://www.weatherapi.com/)
```

In [ ]:
# Etapa 1: Configuração e importação segura da chave de acesso
import requests
from google.colab import userdata

# Obtém a API Key guardada de forma segura nos Secrets do Colab
try:
    API_KEY = userdata.get('WEATHER_API_KEY')
except Exception:
    # Caso não esteja configurado nos Secrets, solicita interativamente
    import getpass
    API_KEY = getpass.getpass("Digite sua API Key da WeatherAPI: ")

# Lista com 10 cidades brasileiras
cidades = [
    "Sao Paulo", "Rio de Janeiro", "Belo Horizonte", "Salvador",
    "Brasilia", "Curitiba", "Fortaleza", "Manaus", "Porto Alegre", "Recife"
]

print(f"Configuração concluída com sucesso para {len(cidades)} cidades!")

In [ ]:
# Etapas 2 e 3: Consulta, validação e extração das informações das cidades
resultados = []
url_base = "http://api.weatherapi.com/v1/current.json"

print("Iniciando requisições...")
for cidade in cidades:
    payload = {'key': API_KEY, 'q': cidade, 'aqi': 'no', 'lang': 'pt'}
    try:
        response = requests.get(url_base, params=payload, timeout=10)
        # Valida o status da resposta HTTP
        response.raise_for_status()

        dados_json = response.json()
        resultados.append(dados_json)

        # Exibe as informações básicas requisitadas no console
        nome = dados_json['location']['name']
        temp = dados_json['current']['temp_c']
        condicao = dados_json['current']['condition']['text']
        print(f"Status: {response.status_code} | {nome}: {temp}°C, {condicao}")

    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP para a cidade '{cidade}': {http_err} (Status: {response.status_code})")
    except Exception as err:
        print(f"Outro erro ocorreu ao consultar '{cidade}': {err}")

print(f"\nTotal de registros coletados com sucesso: {len(resultados)}")

```markdown
## Atividade Prática 02: Análise de Dados com Pandas
```

In [ ]:
# Etapa 1: Conversão para DataFrame e renomeação de colunas
import pandas as pd

lista_estruturada = []
for r in resultados:
    lista_estruturada.append({
        'cidade': r['location']['name'],
        'estado': r['location']['region'],
        'pais': r['location']['country'],
        'temperatura_c': r['current']['temp_c'],
        'sensacao_c': r['current']['feelslike_c'],
        'umidade': r['current']['humidity'],
        'condicao': r['current']['condition']['text'],
        'data_hora_atualizacao': r['current']['last_updated']
    })

df = pd.DataFrame(lista_estruturada)

# Renomeando as colunas para o Português de forma clara
df = df.rename(columns={
    'cidade': 'Cidade',
    'estado': 'Estado',
    'pais': 'País',
    'temperatura_c': 'Temperatura_C',
    'sensacao_c': 'Sensação_Térmica_C',
    'umidade': 'Umidade_Percentual',
    'condicao': 'Condição_Tempo',
    'data_hora_atualizacao': 'Última_Atualização'
})

display(df.head())

In [ ]:
# Etapa 2: Tratamento, tipagem dos dados e resumo estatístico
# Removendo duplicatas caso existam
df = df.drop_duplicates()

# Conversão e tipagem explícita
df['Última_Atualização'] = pd.to_datetime(df['Última_Atualização'])
df['Temperatura_C'] = pd.to_numeric(df['Temperatura_C'])
df['Sensação_Térmica_C'] = pd.to_numeric(df['Sensação_Térmica_C'])
df['Umidade_Percentual'] = pd.to_numeric(df['Umidade_Percentual'])

# Exibe o resumo estatístico numérico
print("\n--- Resumo Estatístico ---")
display(df.describe())

In [ ]:
# Etapa 3: Visualização com Matplotlib e exportação dos dados
import matplotlib.pyplot as plt

# Pergunta: Como variam as temperaturas entre as diferentes cidades brasileiras coletadas?
df_ordenado = df.sort_values(by='Temperatura_C', ascending=False)

plt.figure(figsize=(12, 6))
plt.bar(df_ordenado['Cidade'], df_ordenado['Temperatura_C'], color='skyblue', edgecolor='navy')

plt.title('Comparativo de Temperatura Atual por Cidade no Brasil', fontsize=14, fontweight='bold')
plt.xlabel('Cidade', fontsize=12)
plt.ylabel('Temperatura (°C)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Exportação dos dados para formato CSV
caminho_csv = 'dados_temperatura_cidades.csv'
df.to_csv(caminho_csv, index=False, encoding='utf-8')
print(f"Dados tratados e consolidados exportados com sucesso para: {caminho_csv}")